# 04 · Model Diagnostics

설명용 노트북. 핵심 로직은 `src/credit_risk/{train,evaluate,calibrate,explain}.py`에 있고, 여기서는 저장된 결과(DB `analytics.*`, `reports/model_metrics.json`)와 챔피언 artifact를 불러와 진단만 한다.

> 모델 출력은 Home Credit `TARGET`을 예측한 **proxy PD**이며, 성능 판단은 Holdout 기준으로만 한다.

In [ ]:
import json
import pandas as pd
from credit_risk.config import get_settings
from credit_risk.db import connect, read_frame
from credit_risk.train import load_champion

settings = get_settings()
summary = json.loads((settings.reports_dir / 'model_metrics.json').read_text(encoding='utf-8'))
summary['model_version'], summary['champion'], summary['calibration_method'], summary['threshold']

## Validation 기준 모델 비교 (선택은 Validation ROC-AUC로만)

In [ ]:
pd.DataFrame(summary['validation_metrics_raw']).T[['roc_auc', 'pr_auc', 'brier', 'log_loss', 'ks']]

## Holdout 지표 (최종 1회 평가)

In [ ]:
pd.Series(summary['holdout_calibrated']).round(4)

## Calibration (Holdout, 교정 전/후)

In [ ]:
with connect(settings) as conn:
    calib = read_frame(conn, "SELECT * FROM analytics.calibration_table WHERE split = 'holdout' ORDER BY score_type, bin")
calib[['score_type', 'bin', 'customers', 'mean_pd', 'observed_rate', 'ci_low', 'ci_high', 'small_sample']]

## Lift / gains 와 Exposure-weighted capture

In [ ]:
with connect(settings) as conn:
    lift = read_frame(conn, "SELECT * FROM analytics.lift_table WHERE split = 'holdout' ORDER BY decile")
lift[['decile', 'customers', 'observed_rate', 'lift', 'cum_capture_rate', 'cum_bad_exposure_capture']]

## 세그먼트 진단 (차별의 증명이 아니라 차이의 진단)

In [ ]:
with connect(settings) as conn:
    seg = read_frame(conn, 'SELECT * FROM analytics.segment_performance')
seg[['group_attribute', 'group_value', 'customers', 'observed_rate', 'mean_pd', 'roc_auc', 'recall_at_threshold', 'small_sample_warning']]

## 설명 가능성: 전역 중요도와 개별 reason code
SHAP 값은 모델이 Feature를 사용하는 방식에 대한 **연관** 설명이며 인과 효과가 아니다.

In [ ]:
with connect(settings) as conn:
    imp = read_frame(conn, 'SELECT feature, mean_abs_shap, direction FROM analytics.feature_importance ORDER BY rank LIMIT 15')
    reasons = read_frame(conn, 'SELECT * FROM analytics.reason_codes ORDER BY sk_id_curr, reason_rank LIMIT 9')
imp, reasons

## Artifact 재로딩 & schema 검사

In [ ]:
model = load_champion(settings)
len(model.features), model.feature_schema_hash[:12], model.calibrator.method